In [1]:
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("ggplot")

import cv2

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    cohen_kappa_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score
)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet201, ResNet50, EfficientNetB0
from tensorflow.keras.metrics import Precision, Recall, AUC
from huggingface_hub import hf_hub_download
import zipfile

# MLflow
import mlflow
import mlflow.tensorflow

/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1) Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

zip_path = hf_hub_download(
    repo_id="seyeddd/Solar-PV-Clean-Hotspot-Images",
    repo_type="dataset",
    filename="clean_vs_single_hotspot.zip",
)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("dataset")

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 20

In [4]:
train_dir = "dataset/Dataset/train"
valid_dir = "dataset/Dataset/valid"
test_dir  = "dataset/Dataset/test"

In [5]:
# Output root for artifacts
OUTPUT_ROOT = Path("dl_mlflow_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# MLflow experiment name
MLFLOW_EXPERIMENT = "PV_Hotspot_DL_Model_Comparison"

In [6]:
# 3) Preprocessing
def preprocess(img):
    """
    Your preprocessing:
    - RGB -> Grayscale
    - Gaussian blur
    - Normalize to [0,1]
    - Expand back to 3 channels
    """
    if len(img.shape) == 3:
        img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        img_gray = img

    img_blurred = cv2.GaussianBlur(img_gray, ksize=(3, 3), sigmaX=0)
    img_normalized = img_blurred / 255.0

    if img_normalized.ndim == 2:
        img_normalized = np.stack([img_normalized] * 3, axis=-1)

    return img_normalized

In [7]:
# 4) Generators
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess,
    rotation_range=10,
    shear_range=0.01,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess
)

In [8]:
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=SEED
)

val_gen = val_test_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 2827 images belonging to 2 classes.
Found 700 images belonging to 2 classes.
Found 700 images belonging to 2 classes.


In [9]:
print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")
print(f"Classes: {train_gen.class_indices}")

Training samples: 2827
Validation samples: 700
Test samples: 700
Classes: {'clean': 0, 'single_hotspot': 1}


In [10]:
# Class weights
class_counts = Counter(train_gen.classes)
total_samples = sum(class_counts.values())
class_weights = {cls: total_samples / (len(class_counts) * count) for cls, count in class_counts.items()}
print(f"Class counts: {class_counts}")
print(f"Class weights: {class_weights}")

Class counts: Counter({np.int32(1): 1749, np.int32(0): 1078})
Class weights: {np.int32(0): 1.3112244897959184, np.int32(1): 0.8081761006289309}


In [11]:
# 5) Model builder
def build_transfer_model(base_model, model_name: str, lr: float = 1e-3):
    """
    
    """
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(1, activation="sigmoid")
    ], name=model_name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            Precision(name="precision"),
            Recall(name="recall"),
            AUC(name="auc")
        ]
    )
    return model

In [12]:
def get_models(lr: float = 1e-3):
    """
    3 different backbones.
    """
    densenet = DenseNet201(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    resnet = ResNet50(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    effnet = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))

    return {
        "DenseNet201": build_transfer_model(densenet, "DenseNet201_Model", lr=lr),
        "ResNet50": build_transfer_model(resnet, "ResNet50_Model", lr=lr),
        "EfficientNetB0": build_transfer_model(effnet, "EfficientNetB0_Model", lr=lr),
    }

In [13]:
# 6) Plot + metrics helpers
def save_confusion_matrix(y_true, y_pred, labels, out_path: Path, title: str):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

    plt.figure(figsize=(6, 5))
    disp.plot(cmap=plt.cm.Reds, values_format="d")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_roc_curve(y_true, y_prob, out_path: Path, title: str):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_score = auc(fpr, tpr)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    return roc_score


def save_pr_curve(y_true, y_prob, out_path: Path, title: str):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    return pr_auc


def save_training_curves(history, out_path: Path, model_name: str):
    """
    Single figure with Accuracy/Loss/AUC (train vs val).
    """
    fig = plt.figure(figsize=(16, 4))

    # Accuracy
    ax1 = plt.subplot(1, 3, 1)
    ax1.plot(history.history.get("accuracy", []), label="Train")
    ax1.plot(history.history.get("val_accuracy", []), label="Val")
    ax1.set_title(f"Accuracy - {model_name}")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()

    # Loss
    ax2 = plt.subplot(1, 3, 2)
    ax2.plot(history.history.get("loss", []), label="Train")
    ax2.plot(history.history.get("val_loss", []), label="Val")
    ax2.set_title(f"Loss - {model_name}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()

    # AUC
    ax3 = plt.subplot(1, 3, 3)
    if "auc" in history.history and "val_auc" in history.history:
        ax3.plot(history.history["auc"], label="Train")
        ax3.plot(history.history["val_auc"], label="Val")
        ax3.set_title(f"AUC - {model_name}")
        ax3.set_xlabel("Epoch")
        ax3.set_ylabel("AUC")
        ax3.legend()
    else:
        ax3.text(0.1, 0.5, "AUC history not found", fontsize=12)

    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close(fig)

In [14]:
def evaluate_binary(y_true, y_pred, y_prob):
    """
    Computes a set of metrics (binary).
    """

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }
    return metrics

In [15]:
def get_preds(model, test_gen):
    test_gen.reset()
    y_prob = model.predict(test_gen, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = test_gen.classes.astype(int)
    return y_true, y_prob, y_pred

In [16]:
# 7) Train + MLflow log
def train_one_model(model_name: str, model: tf.keras.Model, lr: float):
    """
    Train one model and log everything to MLflow.
    """
    run_out = OUTPUT_ROOT / model_name
    run_out.mkdir(parents=True, exist_ok=True)

    class_labels = list(test_gen.class_indices.keys())

    # MLflow run
    with mlflow.start_run(run_name=model_name):
        # Log params
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("img_size", f"{IMG_SIZE[0]}x{IMG_SIZE[1]}")
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("optimizer", "Adam")
        mlflow.log_param("learning_rate", lr)

        # Aug params
        mlflow.log_param("rotation_range", 10)
        mlflow.log_param("shear_range", 0.01)
        mlflow.log_param("zoom_range", 0.10)
        mlflow.log_param("horizontal_flip", True)
        mlflow.log_param("fill_mode", "nearest")

        # Train
        earlystop = EarlyStopping(
            monitor="val_auc",
            patience=5,
            restore_best_weights=True,
            mode="max"
        )

        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            class_weight=class_weights,
            callbacks=[earlystop],
            verbose=1
        )

        # Save + log training curves
        curves_path = run_out / "training_curves.png"
        save_training_curves(history, curves_path, model_name)
        mlflow.log_artifact(str(curves_path))

        # Evaluate
        y_true, y_prob, y_pred = get_preds(model, test_gen)

        # classification report
        report = classification_report(y_true, y_pred, target_names=class_labels)
        report_path = run_out / "classification_report.txt"
        report_path.write_text(report)
        mlflow.log_artifact(str(report_path))

        # plots
        cm_path = run_out / "confusion_matrix.png"
        roc_path = run_out / "roc_curve.png"
        pr_path = run_out / "precision_recall_curve.png"

        save_confusion_matrix(y_true, y_pred, class_labels, cm_path, f"Confusion Matrix - {model_name}")
        roc_auc_plot = save_roc_curve(y_true, y_prob, roc_path, f"ROC Curve - {model_name}")
        pr_auc_plot = save_pr_curve(y_true, y_prob, pr_path, f"PR Curve - {model_name}")

        mlflow.log_artifact(str(cm_path))
        mlflow.log_artifact(str(roc_path))
        mlflow.log_artifact(str(pr_path))

        # metrics (sklearn)
        metrics = evaluate_binary(y_true, y_pred, y_prob)
        metrics["roc_auc_plot"] = roc_auc_plot
        metrics["pr_auc_plot"] = pr_auc_plot

        # log metrics to mlflow
        for k, v in metrics.items():
            mlflow.log_metric(k, float(v))

        # Save metrics_summary.csv
        metrics_df = pd.DataFrame({"Value": metrics})
        metrics_csv = run_out / "metrics_summary.csv"
        metrics_df.to_csv(metrics_csv)
        mlflow.log_artifact(str(metrics_csv))

        # Save model
        model_path = run_out / f"{model_name}.keras"
        model.save(model_path)

        # Log model to mlflow
        mlflow.tensorflow.log_model(model, artifact_path="model")

        return metrics

In [17]:
def train_all_models(lr: float = 1e-3):
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    models_dict = get_models(lr=lr)
    all_results = []

    for name, model in models_dict.items():
        print("\n" + "-" * 90)
        print(f"Training + Logging: {name}")
        print("-" * 90)
        metrics = train_one_model(name, model, lr)
        all_results.append({"model": name, **metrics})

    results_df = pd.DataFrame(all_results).sort_values(by="roc_auc", ascending=False)
    results_df.to_csv(OUTPUT_ROOT / "all_models_summary.csv", index=False)
    print("\nSaved:", OUTPUT_ROOT / "all_models_summary.csv")
    print(results_df[["model", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "log_loss", "cohen_kappa"]])

    return results_df

In [18]:
# Run the pipeline
train_all_models(lr=1e-3)

2026/02/14 05:27:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/14 05:27:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/14 05:27:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/14 05:27:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/14 05:27:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/14 05:27:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/14 05:27:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/14 05:27:58 INFO mlflow.store.db.utils: Updating database tables
2026/02/14 05:27:58 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/14 05:27:58 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/14 05:27:58 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/02/14 05:2


------------------------------------------------------------------------------------------
Training + Logging: DenseNet201
------------------------------------------------------------------------------------------


/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/.venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20


/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/.venv/lib/python3.11/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...


Saved: dl_mlflow_outputs/all_models_summary.csv
            model  accuracy  balanced_accuracy  precision  recall        f1  \
0     DenseNet201  0.994286           0.994583   0.997487  0.9925  0.994987   
1        ResNet50  0.845714           0.820000   0.787402  1.0000  0.881057   
2  EfficientNetB0  0.428571           0.500000   0.000000  0.0000  0.000000   

    roc_auc    pr_auc  log_loss  cohen_kappa  
0  0.999967  0.999975  0.012103     0.988343  
1  0.991325  0.993622  0.304338     0.670157  
2  0.855683  0.893622  1.194043     0.000000  


,model,accuracy,balanced_accuracy,precision,recall,f1,cohen_kappa,log_loss,roc_auc,pr_auc,roc_auc_plot,pr_auc_plot
0,DenseNet201,0.994286,0.994583,0.997487,0.9925,0.994987,0.988343,0.012103,0.999967,0.999975,0.999967,0.999975
1,ResNet50,0.845714,0.820000,0.787402,1.0000,0.881057,0.670157,0.304338,0.991325,0.993622,0.991325,0.993614
2,EfficientNetB0,0.428571,0.500000,0.000000,0.0000,0.000000,0.000000,1.194043,0.855683,0.893622,0.855683,0.893455


<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>